In [ ]:
def assign_mechanism_from_pathway(pathway_text: str) -> list[str]:
    """
    Extracts corrosion mechanisms from pathway text using both pathway and mechanism processors.
    This function looks for direct mechanism terms AND infers mechanisms from pathway names.
    """
    found_mechanisms = set()
    
    if not pathway_text:
        return []
    
    # Tokenize the pathway text
    terms_to_process = re.findall(r'\b\w+\b', pathway_text.lower())
    
    # Method 1: Direct mechanism detection
    for term in terms_to_process:
        mechanism = _mechanism_term_processor.find_first_category(term)
        if mechanism:
            found_mechanisms.add(mechanism)
    
    # Method 2: Pathway-to-mechanism inference using pathway processor
    for term in terms_to_process:
        pathway_category = _pathway_processor.find_first_category(term)
        if pathway_category:
            # Map pathway categories to mechanisms (you'll need to define this mapping)
            inferred_mechanisms = infer_mechanisms_from_pathway_category(pathway_category)
            found_mechanisms.update(inferred_mechanisms)
    
    # Method 3: Pattern-based inference for common pathway types
    pathway_lower = pathway_text.lower()
    
    # Sulfur-related pathways
    if any(term in pathway_lower for term in ['sulfur', 'sulfate', 'thiosulfate', 'sulfide']):
        found_mechanisms.add('sulfate_reduction')
    
    # Iron-related pathways
    if any(term in pathway_lower for term in ['iron', 'ferric', 'ferrous', 'heme']):
        found_mechanisms.add('iron_oxidation')
    
    # Organic acid pathways
    if any(term in pathway_lower for term in ['acetate', 'lactate', 'citrate', 'organic acid']):
        found_mechanisms.add('organic_acid_production')
    
    # Biofilm-related pathways
    if any(term in pathway_lower for term in ['biofilm', 'exopolysaccharide', 'quorum']):
        found_mechanisms.add('biofilm_formation')
    
    return list(found_mechanisms)

def assign_mechanism_from_text(text: str) -> list[str]:
    """
    Extracts corrosion mechanisms from any text (enzyme names, descriptions, etc.)
    """
    found_mechanisms = set()
    
    if not text:
        return []
    
    # Tokenize the text
    terms_to_process = re.findall(r'\b\w+\b', text.lower())
    
    for term in terms_to_process:
        mechanism = _mechanism_term_processor.find_first_category(term)
        if mechanism:
            found_mechanisms.add(mechanism)
    
    return list(found_mechanisms)

def infer_mechanisms_from_pathway_category(pathway_category: str) -> list[str]:
    """
    Maps pathway categories to likely corrosion mechanisms.
    This creates the bridge between pathways and mechanisms.
    """
    pathway_to_mechanism_map = {
        'sulfur_metabolism': ['sulfate_reduction', 'sulfur_oxidation'],
        'iron_metabolism': ['iron_oxidation', 'iron_reduction'],
        'organic_acid_metabolism': ['organic_acid_production'],
        'nitrogen_metabolism': ['nitrate_reduction', 'nitrite_reduction'],
        'carbon_metabolism': ['organic_acid_production'],
        'methanogenesis': ['methanogenesis'],
        'biofilm_formation': ['biofilm_formation'],
        'oxygen_consumption': ['aerobic_respiration'],
        'hydrogen_metabolism': ['hydrogen_oxidation'],
        'manganese_processes': ['manganese_oxidation', 'manganese_reduction']
    }
    
    return pathway_to_mechanism_map.get(pathway_category, [])

def enhance_pathway_extraction(record, ec_pathway_mapping, ipath_mapping, ko_ec):
    """
    Enhanced pathway extraction that better integrates all sources and normalizes terms.
    """
    ec_number = record['ec_number']
    
    # Start with existing pathways
    all_pathways = set(record.get('pathways', []))
    
    # 1. Add pathways from EC-pathway mapping with better processing
    if ec_number in ec_pathway_mapping:
        for pathway_id in ec_pathway_mapping[ec_number]:
            # Standardize pathway ID
            std_id = pathway_id
            if pathway_id.startswith('ec'):
                std_id = 'map' + pathway_id[2:]
            
            # Look up pathway name and normalize
            if std_id in pathway_data:
                pathway_name = pathway_data[std_id]
                # Use pathway processor to normalize
                normalized_pathway = _pathway_processor._normalize_term(pathway_name)
                all_pathways.add(pathway_name)  # Keep original for display
    
    # 2. Add ipath data (your ground truth) - this should have priority
    if ec_number in ipath_mapping:
        for ipath_pathway in ipath_mapping[ec_number]:
            # Normalize ipath terms using pathway processor
            if isinstance(ipath_pathway, str):
                normalized_ipath = _pathway_processor._normalize_term(ipath_pathway)
                all_pathways.add(ipath_pathway)
    
    # 3. Add pathways from KO data with normalization
    if ec_number in ko_ec:
        ko_data = ko_ec[ec_number]
        if isinstance(ko_data, list):
            for path in ko_data:
                if isinstance(path, str):
                    all_pathways.add(path)
        elif isinstance(ko_data, dict) and 'pathway' in ko_data:
            path = ko_data['pathway']
            if isinstance(path, str):
                all_pathways.add(path)
    
    # 4. Extract mechanisms from all collected pathways
    pathway_text = ' '.join(all_pathways)
    extracted_mechanisms = assign_mechanism_from_pathway(pathway_text)
    
    # Update record
    record['pathways'] = list(all_pathways)
    existing_mechanisms = record.get('corrosion_mechanisms', [])
    record['corrosion_mechanisms'] = list(set(existing_mechanisms + extracted_mechanisms))
    
    return record

def validate_against_ipath(record, ipath_data):
    """
    Validates detected pathways, mechanisms, and functional categories against ipath ground truth.
    Returns validation metrics and suggestions.
    """
    ec_number = record['ec_number']
    validation_results = {
        'pathway_validation': {},
        'mechanism_validation': {},
        'functional_category_validation': {},
        'overall_confidence': 0.0
    }
    
    if ec_number not in ipath_data:
        validation_results['overall_confidence'] = 0.5  # No ground truth available
        return validation_results
    
    ipath_pathways = ipath_data[ec_number]
    detected_pathways = record.get('pathways', [])
    detected_mechanisms = record.get('corrosion_mechanisms', [])
    detected_fc = [fc['category'] for fc in record.get('functional_categories', [])]
    
    # Pathway validation
    if ipath_pathways and detected_pathways:
        # Normalize both for comparison
        norm_ipath = {_pathway_processor._normalize_term(p) for p in ipath_pathways}
        norm_detected = {_pathway_processor._normalize_term(p) for p in detected_pathways}
        
        overlap = norm_ipath.intersection(norm_detected)
        pathway_precision = len(overlap) / len(norm_detected) if norm_detected else 0
        pathway_recall = len(overlap) / len(norm_ipath) if norm_ipath else 0
        
        validation_results['pathway_validation'] = {
            'precision': pathway_precision,
            'recall': pathway_recall,
            'f1_score': 2 * (pathway_precision * pathway_recall) / (pathway_precision + pathway_recall) if (pathway_precision + pathway_recall) > 0 else 0,
            'overlap_terms': list(overlap)
        }
    
    # Mechanism validation (infer expected mechanisms from ipath)
    if ipath_pathways:
        expected_mechanisms = set()
        for pathway in ipath_pathways:
            expected_mechanisms.update(assign_mechanism_from_pathway(pathway))
        
        if expected_mechanisms and detected_mechanisms:
            norm_expected = {_mechanism_term_processor._normalize_term(m) for m in expected_mechanisms}
            norm_detected = {_mechanism_term_processor._normalize_term(m) for m in detected_mechanisms}
            
            overlap = norm_expected.intersection(norm_detected)
            mech_precision = len(overlap) / len(norm_detected) if norm_detected else 0
            mech_recall = len(overlap) / len(norm_expected) if norm_expected else 0
            
            validation_results['mechanism_validation'] = {
                'precision': mech_precision,
                'recall': mech_recall,
                'f1_score': 2 * (mech_precision * mech_recall) / (mech_precision + mech_recall) if (mech_precision + mech_recall) > 0 else 0,
                'expected_mechanisms': list(expected_mechanisms),
                'overlap_terms': list(overlap)
            }
    
    # Calculate overall confidence
    pathway_f1 = validation_results['pathway_validation'].get('f1_score', 0)
    mechanism_f1 = validation_results['mechanism_validation'].get('f1_score', 0)
    validation_results['overall_confidence'] = (pathway_f1 + mechanism_f1) / 2
    
    return validation_results